# 04 — Full Transformer Training

End-to-end walkthrough: toy English→French dataset, training loop, and translation.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt

import config
from dataset import get_dataloaders, TRANSLATION_PAIRS
from models.transformer import Transformer
from train import compute_loss, set_seed
from visualizations.attention_heatmaps import plot_training_loss

set_seed(config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
train_loader, val_loader, src_vocab, tgt_vocab, _, _ = get_dataloaders()

model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    pad_idx=src_vocab.pad_idx,
    sos_idx=src_vocab.sos_idx,
    eos_idx=src_vocab.eos_idx,
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    d_ff=config.D_FF,
    n_layers=config.N_LAYERS,
    max_seq_len=config.MAX_SEQ_LEN,
    dropout=config.DROPOUT,
).to(device)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, 51):  # shorter for notebook demo
    model.train()
    total = 0.0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        logits, _ = model(src, tgt)
        loss = compute_loss(logits, tgt, src_vocab.pad_idx)
        loss.backward()
        optimizer.step()
        total += loss.item()
    history['train_loss'].append(total / len(train_loader))
    if epoch % 10 == 0:
        print(f'Epoch {epoch}: loss={history["train_loss"][-1]:.4f}')

In [ ]:
fig = plot_training_loss(history['train_loss'])
plt.show()

model.eval()
for en, _ in TRANSLATION_PAIRS[:3]:
    ids = [src_vocab.sos_idx] + src_vocab.encode(en) + [src_vocab.eos_idx]
    ids = ids[:config.MAX_SEQ_LEN] + [src_vocab.pad_idx] * (config.MAX_SEQ_LEN - len(ids))
    src = torch.tensor([ids], device=device)
    out, _ = model.greedy_decode(src, config.MAX_SEQ_LEN)
    print(f'{en} -> {tgt_vocab.decode(out[0].tolist())}')